# NAMESPACES

## Teoría
### 1. El problema
En bash, **todo vive el mismo espacio global**
```bash
log() {
    echo "log 1"
}

# en otro archivo
log() {
    echo "log 2"
}
```
La segunda definición pisa la primera ya que no hay aislamiento.

### 2. Primera solución
Uso de prefijos
```bash
helpers_log() {
    echo "helper"
}

string_to_upper() {
    tr 'a-z' 'A-Z' <<< "$1"
}
```
Funciona, es compatible con POSIX

> Se vuelve feo y largo rápidamente

### 3. Simulación de namespace con `::`
Aquí aparece la convención moderna1:
```bash
helpers::log() {
  printf '[INFO] %s\n' "$*"
}

string::to_upper() {
  tr 'a-z' 'A-Z' <<< "$1"
}
```
Uso:
```bash
helpers::log "hola"
string::to_upper "mundo"
```
#### Qué está pasando realmente
Bash **no entiende namespace**

Simplemente estás definiendo funciones con nombres como:
```bash
"helpers::log"
```
> `::` es solo texto permitido en el nombre.

### 4. Limitaciones
#### No hay encapsulación real
Puedes ser:

```bash
helpers::log
unset -f helpers::log
```

#### No hay privacidad
Todo sigue siendo igual

#### No hay jerarquía real
Esto no funciona:
```bash
helpers::math::sum   # solo es un nombre largo, no estructura real
```

### 5. Módulos + namespace
Combinamos archivos + `::`

#### `helpers.sh`
```bash
helpers::log() {
  printf '[INFO] %s\n' "$*"
}
```

#### `string.sh`
```bash
string::to_upper() {
  tr 'a-z' 'A-Z' <<< "$1"
}
```
> Ya tienes separación lógica + nombres claros


### 6. Añadir control
Usando **`require`**

```bash
require helpers
require string
```
Y usas:
```bash
helpers::log "hola"
```
Ahora tienes:
  * carga controlada
  * nombres organizadas
  * menos riesgo de colisiones

### Regla práctica
Usa `::` cuando:
  * Tu script crece 
  * Tiene múltiples archivos
  * Quieres evitar colisiones
  * Quieres legibilidad tipo API

